In [18]:
# Requirements: pillow, numpy, scikit-image, matplotlib (optional for preview)
from PIL import Image, ImageFilter
import numpy as np

# ---- Config ----
IMG_PATH = "exemplar_object.png"  # change if needed
OUT_DIR  = "../notebooks"

# ---- Load RGBA + alpha ----
img_rgba = Image.open(IMG_PATH).convert("RGBA")
arr = np.array(img_rgba)               # H x W x 4
rgb = arr[..., :3]
# extract alpha channel (0..255 grayscale)
alpha_channel = img_rgba.split()[-1].convert('L')
alpha_np = np.array(alpha_channel)

# ---- 1) Save the alpha map ----
alpha_channel.save(f"{OUT_DIR}/alpha_map.png")

# ---- 2) Border via Gaussian blur (σ=1) ----
# detect object border, interior, and exterior (threshold at 128)
interior_mask = alpha_np > 128   # object interior
exterior_mask = alpha_np <= 128  # background (exterior)

# create a border mask by blurring the alpha and picking transition pixels
blurred_alpha = alpha_channel.filter(ImageFilter.GaussianBlur(radius=1))
blur_np = np.array(blurred_alpha)
border_mask = np.logical_and(blur_np > 0, blur_np < 255)

# ensure mutual exclusivity: prioritize explicit interior/exterior, then border
background_mask = exterior_mask
object_mask = interior_mask

# ---- Checkerboard background for visualization ----
def make_checkerboard(h, w, block=16):
    yy, xx = np.indices((h, w))
    board = ((yy // block) + (xx // block)) % 2
    # two gray levels (RGB)
    return np.where(board[..., None] == 0, 220, 180).astype(np.uint8).repeat(3, axis=2)

H, W = alpha_np.shape
checker = make_checkerboard(H, W, block=16)

def composite(rgb_img, select_mask, checker_bg):
    """Keep original RGB where selected; elsewhere show checkerboard."""
    return np.where(select_mask[..., None], rgb_img, checker_bg).astype(np.uint8)

obj_comp = composite(rgb, object_mask, checker)
bor_comp = composite(rgb, border_mask, checker)
bg_comp  = composite(rgb, background_mask, checker)

# ---- Save visual partitions ----
Image.fromarray(obj_comp).save(f"{OUT_DIR}/partition_object.png")
Image.fromarray(bor_comp).save(f"{OUT_DIR}/partition_border.png")
Image.fromarray(bg_comp).save(f"{OUT_DIR}/partition_background.png")

# ---- Also save the raw masks (8-bit) ----
Image.fromarray((object_mask * 255).astype(np.uint8), mode="L").save(f"{OUT_DIR}/mask_object.png")
Image.fromarray((border_mask * 255).astype(np.uint8), mode="L").save(f"{OUT_DIR}/mask_border.png")
Image.fromarray((background_mask * 255).astype(np.uint8), mode="L").save(f"{OUT_DIR}/mask_background.png")

print("Saved:",
      f"{OUT_DIR}/alpha_map.png",
      f"{OUT_DIR}/partition_object.png",
      f"{OUT_DIR}/partition_border.png",
      f"{OUT_DIR}/partition_background.png",
      sep="\n - ")

Saved:
 - ../notebooks/alpha_map.png
 - ../notebooks/partition_object.png
 - ../notebooks/partition_border.png
 - ../notebooks/partition_background.png


In [19]:
blur_np

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=uint8)

In [20]:
border_mask

array([[False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       ...,
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False]])